# AeroPulse Model 3 — Source Likelihood

**Purpose:** estimate calibrated probabilities for `biomass_burning`, `industrial`, `traffic`, `dust`, `regional_transport`, `mixed_unknown`.

The LLD explicitly says these are likelihoods, not proven causality. Because public source-attribution labels are not supplied by the design, this notebook uses **weak supervision** to create training labels from independent evidence rules. Those labels must be replaced/augmented by an expert-reviewed event catalogue before production claims are made.

In [ ]:
!pip -q install pandas numpy scikit-learn lightgbm joblib requests pyarrow

In [ ]:
# Core
import io
import os
import warnings
from pathlib import Path

import numpy as np
import pandas as pd
import requests

warnings.filterwarnings("ignore")
SEED = 42
np.random.seed(SEED)

DATA_DIR = Path(os.getenv("AEROPULSE_DATA_DIR", "./data"))
ARTIFACT_DIR = Path(os.getenv("AEROPULSE_ARTIFACT_DIR", "./artifacts"))
DATA_DIR.mkdir(parents=True, exist_ok=True)
ARTIFACT_DIR.mkdir(parents=True, exist_ok=True)

REGION = {
    "min_lat": float(os.getenv("AEROPULSE_MIN_LAT", "28.0")),
    "max_lat": float(os.getenv("AEROPULSE_MAX_LAT", "32.8")),
    "min_lon": float(os.getenv("AEROPULSE_MIN_LON", "74.0")),
    "max_lon": float(os.getenv("AEROPULSE_MAX_LON", "78.5")),
}

OPENAQ_API_KEY = os.getenv("OPENAQ_API_KEY", "")
FIRMS_MAP_KEY = os.getenv("FIRMS_MAP_KEY", "")

def http_get_json(url, params=None, headers=None, timeout=60):
    r = requests.get(url, params=params, headers=headers, timeout=timeout)
    r.raise_for_status()
    return r.json()

def safe_float(x):
    try:
        return float(x)
    except Exception:
        return np.nan

def require_env(name):
    value = os.getenv(name)
    if not value:
        raise RuntimeError(f"Missing required environment variable: {name}")
    return value

print("AeroPulse ML environment initialized")
print("Region:", REGION)

In [ ]:
# OpenAQ v3 connector. OpenAQ exposes original measurements plus hourly aggregates.
# Use API key through OPENAQ_API_KEY.
OPENAQ_BASE = "https://api.openaq.org/v3"

def openaq_locations(bbox=REGION, limit=1000):
    params = {
        "bbox": f'{bbox["min_lon"]},{bbox["min_lat"]},{bbox["max_lon"]},{bbox["max_lat"]}',
        "limit": min(limit, 1000),
        "page": 1,
    }
    headers = {"X-API-Key": require_env("OPENAQ_API_KEY")}
    return http_get_json(f"{OPENAQ_BASE}/locations", params=params, headers=headers)

def openaq_location_hours(location_id, date_from, date_to, limit=1000):
    headers = {"X-API-Key": require_env("OPENAQ_API_KEY")}
    params = {
        "datetime_from": pd.Timestamp(date_from, tz="UTC").isoformat(),
        "datetime_to": pd.Timestamp(date_to, tz="UTC").isoformat(),
        "limit": limit,
        "page": 1,
    }
    return http_get_json(f"{OPENAQ_BASE}/locations/{int(location_id)}/hours",
                         params=params, headers=headers)

def discover_pm25_locations(max_locations=100):
    raw = openaq_locations(limit=max_locations)
    rows = raw.get("results", [])
    out = []
    for x in rows:
        params = x.get("parameters", [])
        names = {str(p.get("name", "")).lower() for p in params}
        if "pm25" in names or "pm2.5" in names:
            coords = x.get("coordinates") or {}
            if coords.get("latitude") is not None and coords.get("longitude") is not None:
                out.append({
                    "location_id": x["id"],
                    "name": x.get("name"),
                    "lat": coords["latitude"],
                    "lon": coords["longitude"],
                    "parameters": list(names),
                })
    return pd.DataFrame(out)

def normalize_openaq_hours(payload):
    rows = payload.get("results", [])
    out = []
    for r in rows:
        period = r.get("period", {})
        dt = (period.get("datetimeFrom") or {}).get("utc")
        param = r.get("parameter", {})
        out.append({
            "timestamp": pd.to_datetime(dt, utc=True, errors="coerce"),
            "parameter": str(param.get("name", "")).lower(),
            "value": safe_float(r.get("value")),
            "unit": r.get("unit"),
            "location_id": r.get("locationsId"),
        })
    return pd.DataFrame(out).dropna(subset=["timestamp"])

In [ ]:
# Open-Meteo is used here as a reproducible public weather/ERA5 access path.
# In production, replace this adapter with the IMD connector defined in the AeroPulse LLD.
OPEN_METEO_ARCHIVE = "https://archive-api.open-meteo.com/v1/archive"
OPEN_METEO_FORECAST = "https://api.open-meteo.com/v1/forecast"

WEATHER_VARS = [
    "temperature_2m", "relative_humidity_2m", "surface_pressure",
    "precipitation", "wind_speed_10m", "wind_direction_10m",
    "boundary_layer_height"
]

def weather_archive(lat, lon, start_date, end_date):
    params = {
        "latitude": lat, "longitude": lon,
        "start_date": pd.Timestamp(start_date).date().isoformat(),
        "end_date": pd.Timestamp(end_date).date().isoformat(),
        "hourly": ",".join(WEATHER_VARS),
        "timezone": "UTC",
    }
    return http_get_json(OPEN_METEO_ARCHIVE, params=params)

def weather_forecast(lat, lon, forecast_days=2):
    params = {
        "latitude": lat, "longitude": lon,
        "hourly": ",".join(WEATHER_VARS),
        "forecast_days": forecast_days,
        "timezone": "UTC",
    }
    return http_get_json(OPEN_METEO_FORECAST, params=params)

def normalize_weather(payload):
    h = payload.get("hourly", {})
    df = pd.DataFrame(h)
    if "time" in df:
        df["timestamp"] = pd.to_datetime(df.pop("time"), utc=True)
    return df

In [ ]:
FIRMS_BASE = "https://firms.modaps.eosdis.nasa.gov/api"

def firms_area_csv(source, west, south, east, north, days=1):
    key = require_env("FIRMS_MAP_KEY")
    area = f"{west},{south},{east},{north}"
    url = f"{FIRMS_BASE}/area/csv/{key}/{source}/{area}/{int(days)}"
    r = requests.get(url, timeout=120)
    r.raise_for_status()
    return pd.read_csv(io.StringIO(r.text))

def normalize_firms(df):
    if df.empty:
        return df
    out = df.copy()
    out["timestamp"] = pd.to_datetime(
        out["acq_date"].astype(str) + " " + out["acq_time"].astype(str).str.zfill(4),
        format="%Y-%m-%d %H%M", utc=True, errors="coerce"
    )
    out["frp"] = pd.to_numeric(out.get("frp"), errors="coerce")
    out["confidence_num"] = pd.to_numeric(out.get("confidence"), errors="coerce")
    return out.dropna(subset=["timestamp", "latitude", "longitude"])

In [ ]:
def grid_id(lat, lon, resolution_km=1.0):
    # Approximate 1-km grid using latitude-dependent longitude scaling.
    lat_step = resolution_km / 111.32
    lon_step = resolution_km / (111.32 * np.cos(np.deg2rad(np.clip(lat, -89, 89))))
    ilat = np.floor(lat / lat_step).astype(int)
    ilon = np.floor(lon / lon_step).astype(int)
    return pd.Series(ilat).astype(str) + "_" + pd.Series(ilon).astype(str)

def add_grid(df, lat_col="lat", lon_col="lon"):
    out = df.copy()
    out["grid_id"] = grid_id(out[lat_col].astype(float), out[lon_col].astype(float))
    return out

def haversine_km(lat1, lon1, lat2, lon2):
    r = 6371.0088
    p1, p2 = np.deg2rad(lat1), np.deg2rad(lat2)
    dphi = np.deg2rad(lat2-lat1)
    dl = np.deg2rad(lon2-lon1)
    a = np.sin(dphi/2)**2 + np.cos(p1)*np.cos(p2)*np.sin(dl/2)**2
    return 2*r*np.arcsin(np.sqrt(a))

## 1. Build evidence features

Evidence streams include PM2.5/NO2/CO signatures, wind direction, fire count/FRP, distance to detected fires, and meteorology. Industrial/road density can be added from the canonical grid table when available.

In [ ]:
locations = discover_pm25_locations(max_locations=100)
END = pd.Timestamp.utcnow().floor("h")
START = END - pd.Timedelta(days=int(os.getenv("TRAIN_DAYS", "60")))

records=[]
for _, loc in locations.head(int(os.getenv("MAX_STATIONS","30"))).iterrows():
    try:
        d=normalize_openaq_hours(openaq_location_hours(int(loc.location_id),START,END))
        if not d.empty:
            d["lat"],d["lon"]=loc.lat,loc.lon
            records.append(d)
    except Exception as e:
        print("AQ failed", e)

aq=pd.concat(records,ignore_index=True)
aq["value"]=pd.to_numeric(aq.value,errors="coerce")
aq=aq.dropna(subset=["timestamp","value"])
wide=aq.pivot_table(index=["timestamp","location_id","lat","lon"],columns="parameter",
                    values="value",aggfunc="mean").reset_index()
for c in ["pm25","pm2.5","no2","co","so2","o3"]:
    if c not in wide: wide[c]=np.nan
wide["pm25"]=wide["pm25"].fillna(wide["pm2.5"])
wide["grid_id"]=grid_id(wide.lat,wide.lon)
print(wide.shape)

In [ ]:
weather_parts=[]
for (lat,lon),_ in wide.groupby(["lat","lon"]):
    try:
        weather_parts.append(normalize_weather(weather_archive(lat,lon,START.date(),END.date())).assign(lat=lat,lon=lon))
    except Exception as e: print("weather",e)
weather=pd.concat(weather_parts,ignore_index=True)
wd=np.deg2rad(weather.wind_direction_10m)
weather["wind_u"]=-np.sin(wd)*weather.wind_speed_10m
weather["wind_v"]=-np.cos(wd)*weather.wind_speed_10m
df=wide.merge(weather,on=["timestamp","lat","lon"],how="left")

In [ ]:
# Fire features. If FIRMS is not configured, explicit zeros are used only as a "no fire evidence"
# feature; they must not be interpreted as proof that no fire occurred.
df["fire_count_30km"]=0.0
df["fire_frp_30km"]=0.0
df["fire_min_distance_km"]=np.nan
if FIRMS_MAP_KEY:
    try:
        fire=normalize_firms(firms_area_csv(
            "VIIRS_NOAA20_NRT",REGION["min_lon"],REGION["min_lat"],
            REGION["max_lon"],REGION["max_lat"],days=7))
        if not fire.empty:
            vals=[]
            for _,r in df[["timestamp","lat","lon"]].drop_duplicates().iterrows():
                f=fire[(fire.timestamp<=r.timestamp)&(fire.timestamp>r.timestamp-pd.Timedelta(hours=24))]
                if f.empty:
                    vals.append((r.timestamp,r.lat,r.lon,0,0,np.nan))
                else:
                    dist=haversine_km(r.lat,r.lon,f.latitude,f.longitude)
                    hit=f[dist<=30]
                    vals.append((r.timestamp,r.lat,r.lon,len(hit),hit.frp.sum(),dist.min()))
            ff=pd.DataFrame(vals,columns=["timestamp","lat","lon","fire_count_30km","fire_frp_30km","fire_min_distance_km"])
            df=df.drop(columns=["fire_count_30km","fire_frp_30km","fire_min_distance_km"]).merge(ff,on=["timestamp","lat","lon"],how="left")
    except Exception as e: print("FIRMS disabled",e)
df[["fire_count_30km","fire_frp_30km"]]=df[["fire_count_30km","fire_frp_30km"]].fillna(0)

## 2. Weak-label rules

These rules are deliberately conservative and transparent:

- biomass burning: strong nearby fire evidence + elevated PM2.5
- regional transport: elevated PM2.5 with stronger wind/transport conditions and weak local fire evidence
- traffic: high NO2 relative to PM2.5 with urban/diurnal proxy
- dust: high PM10 would be ideal; if PM10 is unavailable, this class is marked uncertain
- industrial: requires industrial inventory/activity features; otherwise it is not pseudo-labeled
- mixed/unknown: fallback

These labels are **not ground truth**.

In [ ]:
df["hour"]=df.timestamp.dt.hour
df["no2_pm25_ratio"]=df.no2/(df.pm25.clip(lower=1))
df["pm25_high"]=df.pm25>=df.pm25.quantile(.80)
df["fire_signal"]=(df.fire_count_30km>=1) | (df.fire_frp_30km>=20)
df["traffic_signal"]=(df.no2_pm25_ratio>=df.no2_pm25_ratio.quantile(.80)) & df.hour.isin(range(6,11))
df["transport_signal"]=(df.wind_speed_10m>=df.wind_speed_10m.quantile(.60)) & (df.pm25_high)

def weak_label(r):
    if bool(r.fire_signal and r.pm25_high):
        return "biomass_burning"
    if bool(r.traffic_signal):
        return "traffic"
    if bool(r.transport_signal and not r.fire_signal):
        return "regional_transport"
    return "mixed_unknown"

df["source_label"]=df.apply(weak_label,axis=1)
print(df.source_label.value_counts(dropna=False))

In [ ]:
import joblib
from sklearn.calibration import CalibratedClassifierCV
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    balanced_accuracy_score,
    classification_report,
    confusion_matrix,
    log_loss,
)
from sklearn.model_selection import GroupShuffleSplit
from sklearn.pipeline import Pipeline

FEATURES=[
    "pm25","no2","co","so2","o3",
    "no2_pm25_ratio","wind_speed_10m","wind_direction_10m",
    "wind_u","wind_v","temperature_2m","relative_humidity_2m",
    "boundary_layer_height","fire_count_30km","fire_frp_30km",
    "fire_min_distance_km","lat","lon","hour"
]
FEATURES=[c for c in FEATURES if c in df.columns]
data=df.dropna(subset=["source_label"]).copy()

# Group by station to reduce spatial leakage.
gss=GroupShuffleSplit(n_splits=1,test_size=.20,random_state=SEED)
tr_idx,te_idx=next(gss.split(data,groups=data.location_id))
train,test=data.iloc[tr_idx],data.iloc[te_idx]

base=Pipeline([
    ("imputer",SimpleImputer(strategy="median")),
    ("model",LogisticRegression(max_iter=2000,class_weight="balanced",multi_class="auto"))
])
base.fit(train[FEATURES],train.source_label)

raw=base.predict_proba(test[FEATURES])
classes=base.classes_
pred=classes[np.argmax(raw,axis=1)]
print("Balanced accuracy:",balanced_accuracy_score(test.source_label,pred))
print(classification_report(test.source_label,pred,zero_division=0))
print("Log loss:",log_loss(test.source_label,raw,labels=classes))
print("Confusion matrix:\n",confusion_matrix(test.source_label,pred,labels=classes))

## 3. Calibration and artifact

Probability calibration is required by the LLD. A production system should calibrate against independently reviewed source labels; calibration against weak labels only establishes that the pipeline is technically functioning.

In [ ]:
calibrated=CalibratedClassifierCV(base,method="sigmoid",cv=3)
calibrated.fit(train[FEATURES],train.source_label)
proba=calibrated.predict_proba(test[FEATURES])
pred=classes[np.argmax(proba,axis=1)]
print("Calibrated log loss:",log_loss(test.source_label,proba,labels=classes))
print(classification_report(test.source_label,pred,zero_division=0))

bundle={"model":calibrated,"features":FEATURES,"classes":classes.tolist(),
        "label_type":"weak_supervision","version":"1.0.0"}
joblib.dump(bundle,ARTIFACT_DIR/"source_likelihood.joblib")
print("Saved source-likelihood artifact.")

## 4. Live source likelihood

Inference consumes the current canonical grid feature row. The same feature order and preprocessing are used. The output is a probability distribution and top source, never a causal claim.

In [ ]:
bundle=joblib.load(ARTIFACT_DIR/"source_likelihood.joblib")
# Example: score the latest available historical row as a contract test.
row=df.sort_values("timestamp").tail(1)
p=bundle["model"].predict_proba(row[bundle["features"]])[0]
result={c:float(v) for c,v in zip(bundle["classes"],p)}
result["top_source"]=max(result,key=result.get)
result["interpretation"]="LIKELIHOOD_ONLY"
result

### Production validation gate

Do not promote this model to production source attribution until AeroPulse has an independently reviewed event/source dataset. Required release metrics: per-class precision/recall, macro-F1, confusion matrix, log loss/Brier score, calibration curve, geography/season holdouts and a human/expert agreement measure.